# Drive Dosya ID'lerini Dışa Aktar

Bu notebook Drive'daki tüm notebook/sunum/pdf dosyalarının ID'lerini çeker.
Çıktıyı Claude'a yapıştırın — web sayfası linkleri güncellenecek.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
from googleapiclient.discovery import build
from google.colab import auth

auth.authenticate_user()
service = build('drive', 'v3')

DRIVE_KLASOR = "/content/drive/MyDrive/ECS_VB_YZ_90"

def get_file_id(filepath):
    """Drive'daki dosyanın ID'sini bul."""
    # Dosya adını ve parent klasörü kullanarak ara
    filename = os.path.basename(filepath)
    results = service.files().list(
        q=f"name='{filename}' and trashed=false",
        fields="files(id,name,parents)",
        pageSize=10
    ).execute()
    files = results.get('files', [])
    if files:
        return files[0]['id']
    return None

# Tüm dosyaları tara
result = {}

# Notebooks
nb_path = os.path.join(DRIVE_KLASOR, "notebooks")
if os.path.exists(nb_path):
    for week in sorted(os.listdir(nb_path)):
        wp = os.path.join(nb_path, week)
        if not os.path.isdir(wp): continue
        for f in sorted(os.listdir(wp)):
            if not f.endswith('.ipynb'): continue
            fid = get_file_id(os.path.join(wp, f))
            if fid:
                result[f"{week}/{f}"] = fid

# Sunumlar
sunum_path = os.path.join(DRIVE_KLASOR, "sunumlar")
if os.path.exists(sunum_path):
    for f in sorted(os.listdir(sunum_path)):
        if f.endswith(('.pptx', '.pdf')):
            fid = get_file_id(os.path.join(sunum_path, f))
            if fid:
                result[f"sunumlar/{f}"] = fid

# İzlence
izl = os.path.join(DRIVE_KLASOR, "egitim-izlencesi.md")
if os.path.exists(izl):
    fid = get_file_id(izl)
    if fid:
        result["egitim-izlencesi.md"] = fid

print(f"Toplam {len(result)} dosya bulundu.")
print()
print("=== BAŞLANGIÇ ===")
print(json.dumps(result, indent=2, ensure_ascii=False))
print("=== BİTİŞ ===")
print()
print("Yukarıdaki JSON çıktısını kopyalayıp Claude'a yapıştırın.")